### Step 1: Mount the Google Drive

Remember to use GPU runtime before mounting your Google Drive. (Runtime --> Change runtime type).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Open the project directory

Ensure the dataset is in `emg2qwerty/data/` (with .hdf5 files). Adjust the path if your project is elsewhere.

In [ ]:
%cd /content/drive/MyDrive/emg2qwerty

### Step 2a: Verify dataset

Place the dataset in `emg2qwerty/data/` (with .hdf5 session files inside) before running. The config expects `data/{session_name}.hdf5`.

In [ ]:
import os

data_dir = 'data'
if os.path.exists(data_dir):
    hdf5_files = [f for f in os.listdir(data_dir) if f.endswith('.hdf5')]
    if hdf5_files:
        print(f"Dataset found: {len(hdf5_files)} .hdf5 files in {data_dir}/")
    else:
        print(f"WARNING: {data_dir}/ exists but has no .hdf5 files. Add session files (e.g. *.hdf5) to {data_dir}/")
else:
    print(f"Place the dataset in {os.path.join(os.getcwd(), data_dir)}/")
    print("Expected: data/session_name.hdf5 (files from single_user config)")

### Step 3: Install required packages

After installing them, Colab will require you to restart the session.

In [ ]:
!pip install -r requirements.txt

### Step 4: Start your experiments!

- The dataset is in `data/` (place .hdf5 files in `emg2qwerty/data/` before running).
- You may now start your experiments with any scripts! Below are examples of single-user training and testing (greedy decoding).
- **There are two ways to track the logs:**
  - 1. Keep `--multirun`, and the logs will not be printed here, but they will be saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/submitit_logs/`.
  - 2. Comment out `--multirun` and the logs will be printed in this notebook, but they will not be saved.

#### Training

- The checkpoints are saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/checkpoints/`.
- **Model options:** Use `model=tds_conv_ctc` (baseline) or `model=transformer_ctc` (transformer).

In [ ]:
# Option A: TDS Conv baseline (default)
# !python -m emg2qwerty.train \
#   user="single_user" \
#   model=tds_conv_ctc \
#   trainer.accelerator=gpu trainer.devices=1 \
#   # --multirun

# Option B: Transformer baseline
!python -m emg2qwerty.train \
  user="single_user" \
  model=transformer_ctc \
  trainer.accelerator=gpu trainer.devices=1 \
  # --multirun

#### Testing

- Replace `Your_Path_to_Checkpoint` with your checkpoint path.
- Use `model=transformer_ctc` when evaluating a transformer checkpoint; use `model=tds_conv_ctc` (or omit) for the TDS baseline.

In [ ]:
# Testing (use model=transformer_ctc for transformer checkpoints)
!python -m emg2qwerty.train \
  user="single_user" \
  model=transformer_ctc \
  checkpoint="Your_Path_to_Checkpoint" \
  train=False trainer.accelerator=gpu \
  decoder=ctc_greedy \
  hydra.launcher.mem_gb=64 \
  # --multirun